# 第2回 発展宿題: 用語集のLoRA学習と検索を比較する

`lora_format_tuning.ipynb` のモデル・LoRA設定・Colab用ライブラリを引き継ぎ、FAQを講座用語集に置き換えた版です。
**T4 GPUを選び、上から順に実行してください。** 外部APIキーやリポジトリのcloneは不要です。

1. 同梱した `scripts/make_sft_dataset.py` を実際に実行し、58語・116件の `messages` JSONLを生成します。
2. 学習前後のモデルには**質問だけ**を渡します（用語集本文や正解を渡しません）。回答は生成スクリプトと同じ定義文で、元演習のJSON形式は要求しません。
3. 手作りの22問（掲載語8・言い換え8・未掲載語6）を、学習前・学習後・用語集検索の3方式で比較します。
4. 未掲載語への回答、言い換えへの対応、学習時間・検索の出典を見て考察します。

**実行後に共有するもの**: 最後の **[HW-7] 結果の保存と共有** でダウンロードする `hw_evidence.zip`。
添付できない場合は、同セルの `HW_REPORT_BEGIN` から `HW_REPORT_END` までをコピーしてください。
全質問の出力・正解・自動採点を含むので、これだけで考察の根拠を確認できます。
結果を見てから `lectures/02/answer.md` を簡潔に完成させます。

設定を変えて再実行する場合は、**セッションを再起動して先頭から**実行します。学習済みadapterを付けたまま再学習しないでください。

## 0. 環境の確認と導入

元ノートブックと同じT4用の固定版です。PyTorchはColab同梱版を使います。
GPUが無い場合は「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選びます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定と入力ファイルの復元

用語集・生成スクリプト・検索実装の**この作業時点のコピー**を埋め込んでいます。
Colab上で元と同じディレクトリ構成に復元し、その生成スクリプトを実行します。
元ノートブックの14件×20周に対して、今回は116件×3周（実効バッチ8、約45更新）です。
データの長さが異なるため、元演習の所要時間と同じとは限りません。

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, math, platform, subprocess, sys, time, unicodedata
import pandas as pd
from IPython.display import display

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 3,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "train_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "gen": {"do_sample": False},
    "work_dir": "/content/glossary_lora_hw",
}
WORK = Path(CONFIG["work_dir"])
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK / "source"
SYSTEM = (
    "あなたはAIE2026の用語説明係です。講座の用語集に沿って、質問された用語の定義を簡潔な日本語で答えてください。"
    "用語集にない語は『用語集には記載がありません。確認が必要です。』と答えてください。"
)
UNKNOWN_ANSWER = "用語集には記載がありません。確認が必要です。"
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

In [ ]:
# 元ファイルの内容。通常は編集不要です。
EMBEDDED_FILES = {'data/glossary.json': '{\n  "version": "2026-draft-1",\n  "note": "AIE2026 第1〜3回・第5回の用語集（たたき台）。講義スライド確定後に内容を差し替える。",\n  "terms": [\n    {\n      "id": "ai-engineering",\n      "term": "AIエンジニアリング",\n      "aliases": ["AI Engineering"],\n      "lecture": 1,\n      "category": "全体像",\n      "definition": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。",\n      "related": ["llm", "evaluation-loop"]\n    },\n    {\n      "id": "evaluation-loop",\n      "term": "評価と改善のサイクル",\n      "aliases": ["改善サイクル"],\n      "lecture": 1,\n      "category": "全体像",\n      "definition": "作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。",\n      "related": ["ai-engineering", "hallucination"]\n    },\n    {\n      "id": "llm",\n      "term": "LLM（大規模言語モデル）",\n      "aliases": ["大規模言語モデル", "Large Language Model"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。",\n      "related": ["token", "prompt", "pretraining"]\n    },\n    {\n      "id": "token",\n      "term": "トークン",\n      "aliases": ["token"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。",\n      "related": ["context-window", "rate-limit"]\n    },\n    {\n      "id": "context-window",\n      "term": "コンテキストウィンドウ",\n      "aliases": ["コンテキスト長", "context window"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。",\n      "related": ["token", "agent-loop"]\n    },\n    {\n      "id": "prompt",\n      "term": "プロンプト",\n      "aliases": ["prompt"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。",\n      "related": ["system-prompt", "llm"]\n    },\n    {\n      "id": "system-prompt",\n      "term": "システムプロンプト",\n      "aliases": ["指示文", "instructions", "system instruction"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。",\n      "related": ["prompt", "guardrail"]\n    },\n    {\n      "id": "temperature",\n      "term": "temperature",\n      "aliases": ["温度パラメータ"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。",\n      "related": ["llm", "evaluation-loop"]\n    },\n    {\n      "id": "hallucination",\n      "term": "ハルシネーション",\n      "aliases": ["幻覚", "hallucination"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。",\n      "related": ["tool", "evaluation-loop"]\n    },\n    {\n      "id": "api",\n      "term": "API",\n      "aliases": ["Application Programming Interface", "Web API"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。",\n      "related": ["api-key", "rate-limit"]\n    },\n    {\n      "id": "api-key",\n      "term": "APIキー",\n      "aliases": ["API key"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。",\n      "related": ["codespaces-secret", "environment-variable"]\n    },\n    {\n      "id": "environment-variable",\n      "term": "環境変数",\n      "aliases": ["environment variable", "env"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。",\n      "related": ["api-key", "codespaces-secret"]\n    },\n    {\n      "id": "rate-limit",\n      "term": "レート制限",\n      "aliases": ["rate limit", "429エラー"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。",\n      "related": ["api", "agent-loop", "default-model"]\n    },\n    {\n      "id": "codespaces",\n      "term": "GitHub Codespaces",\n      "aliases": ["Codespaces", "コードスペース"],\n      "lecture": 1,\n      "category": "開発環境",\n      "definition": "GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。",\n      "related": ["devcontainer", "codespaces-secret"]\n    },\n    {\n      "id": "devcontainer",\n      "term": "devcontainer",\n      "aliases": ["Dev Container", "devcontainer.json"],\n      "lecture": 1,\n      "category": "開発環境",\n      "definition": "開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。",\n      "related": ["codespaces"]\n    },\n    {\n      "id": "codespaces-secret",\n      "term": "Codespaces secret",\n      "aliases": ["Codespacesのシークレット"],\n      "lecture": 1,\n      "category": "開発環境",\n      "definition": "GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。",\n      "related": ["api-key", "environment-variable", "codespaces"]\n    },\n    {\n      "id": "fork",\n      "term": "fork",\n      "aliases": ["フォーク"],\n      "lecture": 1,\n      "category": "Git/GitHub",\n      "definition": "他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。",\n      "related": ["branch", "pull-request"]\n    },\n    {\n      "id": "branch",\n      "term": "ブランチ",\n      "aliases": ["branch"],\n      "lecture": 1,\n      "category": "Git/GitHub",\n      "definition": "作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。",\n      "related": ["pull-request", "fork"]\n    },\n    {\n      "id": "pull-request",\n      "term": "プルリクエスト（PR）",\n      "aliases": ["PR", "Pull Request"],\n      "lecture": 1,\n      "category": "Git/GitHub",\n      "definition": "あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。",\n      "related": ["branch", "answer-md"]\n    },\n    {\n      "id": "github-copilot",\n      "term": "GitHub Copilot",\n      "aliases": ["Copilot"],\n      "lecture": 1,\n      "category": "AIコーディング",\n      "definition": "GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。",\n      "related": ["code-completion", "coding-agent"]\n    },\n    {\n      "id": "code-completion",\n      "term": "コード補完",\n      "aliases": ["インライン補完", "inline suggestion"],\n      "lecture": 1,\n      "category": "AIコーディング",\n      "definition": "書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。",\n      "related": ["github-copilot"]\n    },\n    {\n      "id": "coding-agent",\n      "term": "コーディングエージェント",\n      "aliases": ["Coding Agent", "AIコーディングエージェント"],\n      "lecture": 1,\n      "category": "AIコーディング",\n      "definition": "ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。",\n      "related": ["ai-agent", "harness"]\n    },\n    {\n      "id": "pretraining",\n      "term": "事前学習",\n      "aliases": ["pre-training", "プレトレーニング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。",\n      "related": ["fine-tuning", "llm"]\n    },\n    {\n      "id": "fine-tuning",\n      "term": "ファインチューニング",\n      "aliases": ["fine-tuning", "追加学習", "FT"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。",\n      "related": ["sft", "peft", "lora"]\n    },\n    {\n      "id": "sft",\n      "term": "教師ありファインチューニング（SFT）",\n      "aliases": ["SFT", "Supervised Fine-Tuning", "指示チューニング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。",\n      "related": ["fine-tuning", "lora"]\n    },\n    {\n      "id": "peft",\n      "term": "PEFT",\n      "aliases": ["Parameter-Efficient Fine-Tuning", "パラメータ効率の良いファインチューニング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。",\n      "related": ["lora", "fine-tuning"]\n    },\n    {\n      "id": "lora",\n      "term": "LoRA",\n      "aliases": ["Low-Rank Adaptation"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。",\n      "related": ["peft", "lora-rank", "qlora"]\n    },\n    {\n      "id": "lora-rank",\n      "term": "ランク（LoRAのr）",\n      "aliases": ["rank", "r"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。",\n      "related": ["lora"]\n    },\n    {\n      "id": "qlora",\n      "term": "QLoRA",\n      "aliases": ["Quantized LoRA"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。",\n      "related": ["lora", "quantization"]\n    },\n    {\n      "id": "quantization",\n      "term": "量子化",\n      "aliases": ["quantization"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。",\n      "related": ["qlora"]\n    },\n    {\n      "id": "learning-rate",\n      "term": "学習率",\n      "aliases": ["learning rate", "lr"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。",\n      "related": ["epoch", "overfitting"]\n    },\n    {\n      "id": "epoch",\n      "term": "エポック",\n      "aliases": ["epoch"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。",\n      "related": ["overfitting", "learning-rate"]\n    },\n    {\n      "id": "overfitting",\n      "term": "過学習",\n      "aliases": ["overfitting", "オーバーフィッティング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。",\n      "related": ["epoch", "evaluation-loop"]\n    },\n    {\n      "id": "huggingface-hub",\n      "term": "Hugging Face Hub",\n      "aliases": ["Hugging Face", "HF Hub"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。",\n      "related": ["lora", "pretraining"]\n    },\n    {\n      "id": "ai-agent",\n      "term": "AIエージェント",\n      "aliases": ["エージェント", "agent"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。",\n      "related": ["agent-loop", "tool", "harness"]\n    },\n    {\n      "id": "agent-loop",\n      "term": "エージェントループ",\n      "aliases": ["agent loop", "ツール呼び出しループ"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。",\n      "related": ["ai-agent", "tool", "rate-limit"]\n    },\n    {\n      "id": "tool",\n      "term": "ツール（Tool use / Function calling）",\n      "aliases": ["ツール呼び出し", "Function calling", "Tool use", "ツール"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。",\n      "related": ["tool-description", "agent-loop"]\n    },\n    {\n      "id": "tool-description",\n      "term": "ツールの説明文",\n      "aliases": ["docstring", "ツールスキーマ", "tool schema"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。",\n      "related": ["tool", "pydantic-ai"]\n    },\n    {\n      "id": "structured-output",\n      "term": "構造化出力",\n      "aliases": ["structured output", "JSON出力"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。",\n      "related": ["pydantic-ai"]\n    },\n    {\n      "id": "harness",\n      "term": "ハーネス",\n      "aliases": ["agent harness", "エージェントハーネス"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。",\n      "related": ["coding-agent", "pi"]\n    },\n    {\n      "id": "pi",\n      "term": "Pi",\n      "aliases": ["Pi coding agent", "pi agent"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。",\n      "related": ["harness", "coding-agent"]\n    },\n    {\n      "id": "pydantic-ai",\n      "term": "Pydantic AI",\n      "aliases": ["pydantic-ai", "PydanticAI"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。",\n      "related": ["test-model", "structured-output", "adk"]\n    },\n    {\n      "id": "adk",\n      "term": "ADK（Agent Development Kit）",\n      "aliases": ["ADK", "Agent Development Kit", "google-adk"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。",\n      "related": ["pydantic-ai", "ai-agent"]\n    },\n    {\n      "id": "mcp",\n      "term": "MCP（Model Context Protocol）",\n      "aliases": ["MCP", "Model Context Protocol"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。",\n      "related": ["tool", "ai-agent", "mcp-server", "mcp-client"]\n    },\n    {\n      "id": "guardrail",\n      "term": "ガードレール",\n      "aliases": ["guardrail"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。",\n      "related": ["human-in-the-loop", "system-prompt"]\n    },\n    {\n      "id": "human-in-the-loop",\n      "term": "Human-in-the-loop",\n      "aliases": ["HITL", "人間による承認", "確認を挟む"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。",\n      "related": ["guardrail", "tool"]\n    },\n    {\n      "id": "test-model",\n      "term": "TestModel",\n      "aliases": ["テスト用モデル"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。",\n      "related": ["pydantic-ai", "rate-limit"]\n    },\n    {\n      "id": "homework-levels",\n      "term": "宿題の3レベル",\n      "aliases": ["3レベル", "宿題レベル"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。",\n      "related": ["answer-md"]\n    },\n    {\n      "id": "answer-md",\n      "term": "answer.md",\n      "aliases": ["解答ファイル"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。",\n      "related": ["pull-request", "homework-levels"]\n    },\n    {\n      "id": "lec-start-tag",\n      "term": "lecNN-start タグ",\n      "aliases": ["開始タグ", "lec03-start"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。",\n      "related": ["branch"]\n    },\n    {\n      "id": "default-model",\n      "term": "既定モデル",\n      "aliases": ["デフォルトモデル", "gemini-3.5-flash-lite"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。",\n      "related": ["rate-limit", "api-key"]\n    },\n    {\n      "id": "external-tool",\n      "term": "外部ツール",\n      "aliases": ["external tool", "外部サービス"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。",\n      "related": ["tool", "mcp", "mcp-server"]\n    },\n    {\n      "id": "mcp-server",\n      "term": "MCPサーバー",\n      "aliases": ["MCP server"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。",\n      "related": ["mcp", "mcp-client", "mcp-transport", "fetch-server"]\n    },\n    {\n      "id": "mcp-client",\n      "term": "MCPクライアント",\n      "aliases": ["MCP client", "MCPホスト"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。",\n      "related": ["mcp", "mcp-server", "mcp-config"]\n    },\n    {\n      "id": "mcp-transport",\n      "term": "トランスポート（stdio・Streamable HTTP）",\n      "aliases": ["stdio", "Streamable HTTP", "transport", "MCPのトランスポート"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。",\n      "related": ["mcp-server", "mcp-client"]\n    },\n    {\n      "id": "mcp-config",\n      "term": "mcpServers設定",\n      "aliases": ["mcpServers", "MCPの設定ファイル", "mcp_servers.json", "mcp.json"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。",\n      "related": ["mcp-client", "tool-namespace"]\n    },\n    {\n      "id": "tool-namespace",\n      "term": "ツールの名前空間",\n      "aliases": ["ツール名のプレフィックス", "prefix", "名前の衝突"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。",\n      "related": ["mcp-config", "tool-description", "rate-limit"]\n    },\n    {\n      "id": "fetch-server",\n      "term": "fetchサーバー",\n      "aliases": ["mcp-server-fetch", "fetch", "Web取得ツール"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。",\n      "related": ["mcp-server", "external-tool", "guardrail"]\n    }\n  ]\n}\n', 'scripts/make_sft_dataset.py': '"""第2回 宿題（発展）: 用語集から SFT 用の Q&A データ（JSONL）を作る。\n\n使い方:\n    uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl\n\nできたファイルを Colab にアップロードし、LoRA 学習のデータとして使う。\n質問の言い回しを増やす、別名でも質問を作る、検証用に一部を分ける、などを工夫してみよう。\n"""\n\nimport json\nimport sys\nfrom pathlib import Path\n\nGLOSSARY = Path(__file__).resolve().parent.parent / "data" / "glossary.json"\nTEMPLATES = ["「{term}」とは何ですか？", "{term}について簡単に説明してください。"]\n\n\ndef main() -> None:\n    terms = json.loads(GLOSSARY.read_text(encoding="utf-8"))["terms"]\n    for t in terms:\n        for template in TEMPLATES:\n            record = {\n                "messages": [\n                    {"role": "user", "content": template.format(term=t["term"])},\n                    {"role": "assistant", "content": t["definition"]},\n                ]\n            }\n            sys.stdout.write(json.dumps(record, ensure_ascii=False) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n', 'app/tools.py': '"""第3回: エージェントに持たせるツール。\n\nここに書いた関数の docstring と型ヒントが、そのままモデルへの「ツールの説明文」になる。\n説明文を書き換えると、モデルがツールを使うかどうか・どんな引数で呼ぶかが変わる。\n"""\n\nimport json\nfrom functools import cache\nfrom pathlib import Path\n\nDATA_DIR = Path(__file__).resolve().parent.parent / "data"\nMAX_RESULTS = 3\n\n\n@cache\ndef load_glossary() -> list[dict]:\n    with open(DATA_DIR / "glossary.json", encoding="utf-8") as f:\n        return json.load(f)["terms"]\n\n\ndef _normalize(text: str) -> str:\n    return text.casefold().replace(" ", "").replace("\u3000", "")\n\n\ndef _names(entry: dict) -> list[str]:\n    return [_normalize(entry["term"]), _normalize(entry["id"])] + [\n        _normalize(a) for a in entry["aliases"]\n    ]\n\n\ndef search_glossary(query: str) -> str:\n    """講座の用語集から用語を検索し、定義と、どの回で扱うかを返す。\n\n    AIエンジニアリングの用語や、この講座（AIE2026）に固有の言葉について\n    答える前に使うこと。query には調べたい用語を1つだけ入れる（例: "LoRA"）。\n    """\n    q = _normalize(query)\n    if not q:\n        return "検索語が空です。調べたい用語を1つ指定してください。"\n    glossary = load_glossary()\n\n    exact = [e for e in glossary if q in _names(e)]\n    # 「r」のような短い別名が何にでも当たらないよう、名前側が短いものは除く\n    partial = [\n        e for e in glossary\n        if e not in exact and any(q in n or (len(n) >= 3 and n in q) for n in _names(e))\n    ]\n    in_definition = [\n        e for e in glossary\n        if e not in exact and e not in partial and q in _normalize(e["definition"])\n    ]\n    hits = (exact + partial + in_definition)[:MAX_RESULTS]\n\n    if not hits:\n        return f"用語集に「{query}」は見つかりませんでした。"\n    return json.dumps(\n        [\n            {\n                "term": e["term"],\n                "lecture": e["lecture"],\n                "definition": e["definition"],\n                "related": e["related"],\n            }\n            for e in hits\n        ],\n        ensure_ascii=False,\n    )\n\n\ndef get_lecture_summary(lecture: int) -> str:\n    """指定した回（1始まりの整数）の講義の要点を返す。\n\n    「第3回では何を学ぶ？」のように、回の内容や関係を聞かれたときに使う。\n    """\n    path = DATA_DIR / "lectures" / f"{lecture:02d}.md"\n    if not path.exists():\n        available = sorted(p.stem for p in (DATA_DIR / "lectures").glob("*.md"))\n        return f"第{lecture}回の要点はまだありません。用意されている回: {\', \'.join(available)}"\n    return path.read_text(encoding="utf-8")\n'}

for relative, content in EMBEDDED_FILES.items():
    path = SOURCE / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
INPUT_HASHES = {name: hashlib.sha256(content.encode("utf-8")).hexdigest()
                for name, content in EMBEDDED_FILES.items()}
glossary = json.loads((SOURCE / "data/glossary.json").read_text(encoding="utf-8"))
terms = glossary["terms"]
by_id = {entry["id"]: entry for entry in terms}

# リダイレクトの文字コード差を避け、UTF-8の標準出力をbytesでそのまま保存する。
import os
completed = subprocess.run(
    [sys.executable, str(SOURCE / "scripts/make_sft_dataset.py")],
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}, capture_output=True, check=True,
)
SFT_PATH = WORK / "glossary_sft.jsonl"
SFT_PATH.write_bytes(completed.stdout)
records = [json.loads(line) for line in SFT_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(records) == len(terms) * 2
assert all([m["role"] for m in r["messages"]] == ["user", "assistant"] for r in records)
assert all(isinstance(m["content"], str) and m["content"] for r in records for m in r["messages"])

# 定義文は変えず、全レコードに推論時と同じsystem指示を付ける。
train_records = [{"messages": [{"role": "system", "content": SYSTEM}] + r["messages"]} for r in records]
# 第3の質問テンプレートで検証する。語と定義自体は学習済みで、未知の知識の検証ではない。
valid_records = [{"messages": [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"講座の用語として「{entry['term']}」の定義を述べてください。"},
    {"role": "assistant", "content": entry["definition"]},
]} for entry in terms]
print(f"用語数={len(terms)}, train={len(train_records)}, valid={len(valid_records)}")
print("生成JSONLの先頭:", json.dumps(records[0], ensure_ascii=False))
print("学習に実際に渡す先頭:", json.dumps(train_records[0], ensure_ascii=False))

## 2. 評価問題と正解 [HW-2]

下の問題は学習用の2テンプレートと異なる文です。**掲載語の定義は学習に使います**。
従って、測るのは「学習済みの知識を別の質問で使えるか」であり、未知の知識の獲得ではありません。
言い換え8問には、別名を含む質問と、用語名を直接書かない説明からの質問を含めています。
未掲載6問には、実在する一般用語と架空の語の両方を入れます。

正解は用語集の定義文、未掲載語は確認要求です。内容の自動採点は、下に手書きした
**必須概念の語句グループ（各グループ内はOR、グループ間はAND）**による近似です。
意味が正しくても落ちたり、否定や矛盾を検出できない場合があるため、最後に原文を確認します。
未掲載の実在語への説明は、それだけで事実誤りとは判定せず、**用語集の範囲外への回答**として数えます。

In [ ]:
# id, 掲載語を含む質問, 言い換えた質問, 必須概念の語句グループ
CASES = [
    ("lora", "LoRAはどのような学習手法ですか？元の重みと更新分の扱いを含めて説明して。",
     "Low-Rank Adaptationは、元のモデルをどのように追加学習する方式ですか？",
     [["固定", "凍結"], ["低ランク", "小さな行列"], ["学習", "更新"]]),
    ("qlora", "QLoRAはGPUメモリをどう節約して学習しますか？",
     "Quantized LoRAでは、ベースモデルをどう扱いますか？",
     [["量子化", "4ビット", "4bit"], ["LoRA"], ["メモリ", "節約"]]),
    ("sft", "教師ありファインチューニング（SFT）に必要なデータと目的を教えて。",
     "質問と模範回答のペアを見せて、望ましい応答を覚えさせる追加学習はどのような方法ですか？",
     [["入力", "質問"], ["出力", "回答"], ["ペア", "対"], ["学習"]]),
    ("quantization", "量子化ではモデルの何を変えて、何を減らしますか？",
     "モデルの重みを少ないビット数で表す技術の仕組みと利点を説明して。",
     [["重み"], ["ビット", "bit"], ["メモリ", "計算量"]]),
    ("overfitting", "過学習とはどのような状態ですか？学習データと未知の入力への性能の違いも説明して。",
     "練習したデータに合わせすぎて、初めての入力にうまく答えられなくなる状態を説明して。",
     [["学習データ"], ["過度", "適合", "合わせすぎ", "覚え"], ["未知", "汎化", "未見"], ["落ち", "低下", "悪化"]]),
    ("context-window", "コンテキストウィンドウとは何の上限ですか？入力と出力の数え方も説明して。",
     "コンテキスト長は、1回の呼び出しで扱える量をどう制限するものですか？",
     [["入力"], ["出力"], ["合計", "合わせ"], ["トークン"], ["上限"]]),
    ("homework-levels", "宿題の3レベルの名前と、完了の条件を教えて。",
     "この講座の課題にはどんな難しさの選択肢があり、いくつ選べば終わりですか？",
     [["基礎"], ["実装"], ["発展"], ["どれか1", "いずれか1", "1つ", "一つ"]]),
    ("lec-start-tag", "lecNN-start タグは、どのようなときに使いますか？",
     "前の回を終えていなくても、各回の演習の出発点から作業を始めるには何を利用しますか？",
     [["タグ"], ["開始", "始め"], ["ブランチ"]]),
]
test = []
for term_id, named_q, paraphrase_q, groups in CASES:
    for category, question in [("known", named_q), ("paraphrase", paraphrase_q)]:
        test.append({"id": f"{category}-{term_id}", "category": category,
                     "question": question, "term_id": term_id,
                     "reference": by_id[term_id]["definition"], "keyword_groups": groups,
                     "expected_abstain": False})
UNKNOWN_TERMS = ["知識蒸留", "勾配降下法", "正則化", "AIE2026認定スコア", "AIE2026専用推論チップ", "量子ドロップLoRA"]
for i, term in enumerate(UNKNOWN_TERMS, 1):
    test.append({"id": f"unknown-{i}", "category": "unknown",
                 "question": f"「{term}」とは何ですか？", "term_id": None,
                 "reference": UNKNOWN_ANSWER, "keyword_groups": [], "expected_abstain": True})

def normalize(text):
    return "".join(unicodedata.normalize("NFKC", text).casefold().split())

def detects_abstention(raw):
    # 表層の自動判定。説明と拒否が混ざる場合などは原文で確認する。
    text = normalize(raw)
    negatives = ["記載がありません", "記載されていません", "載っていません", "載っていない",
                 "含まれていません", "見つかりません", "登録されていません", "未掲載", "用語集にはない", "用語集にない"]
    return "用語集" in text and any(normalize(word) in text for word in negatives)

def grade(raw, ex):
    abstained = detects_abstention(raw)
    missing = [group for group in ex["keyword_groups"]
               if not any(normalize(word) in normalize(raw) for word in group)]
    content_ok = abstained if ex["expected_abstain"] else (not abstained and not missing)
    return {"content_ok": bool(content_ok), "abstained": bool(abstained), "missing_concepts": missing}

train_questions = {r["messages"][1]["content"] for r in train_records}
valid_questions = {r["messages"][1]["content"] for r in valid_records}
questions = [e["question"] for e in test]
assert len(set(questions)) == len(questions) == 22
assert not (set(questions) & train_questions), "評価質問が学習に混入しています"
assert not (set(questions) & valid_questions), "評価質問が検証に混入しています"
all_names = {normalize(name) for entry in terms for name in [entry["id"], entry["term"], *entry["aliases"]]}
assert all(normalize(term) not in all_names for term in UNKNOWN_TERMS)
assert all(grade(ex["reference"], ex)["content_ok"] for ex in test), "採点基準が正解文と不整合です"
(WORK / "eval_questions.json").write_text(json.dumps(test, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([{"id": e["id"], "区分": e["category"], "質問": e["question"], "正解": e["reference"]} for e in test]))
print("train / validとの質問重複なし。未掲載語の名称一致なし。正解文の採点チェックOK。")

## 3. 学習前の評価 [HW-3]

元ノートブックと同じモデル・revision・32bitを使います。
最初に全学習・検証データが512トークンに収まることを確認し、回答が切り捨てられる事故を防ぎます。
学習前後とも同じsystem指示・貪欲生成で、用語集本文を渡さず22問に答えさせます。
`truncated=True` は生成上限に達した可能性がある回答です（採点と別に原文を確認）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
assert torch.cuda.is_available(), "T4 GPUを選択してから、先頭から実行してください"
DEVICE = "cuda"
set_seed(CONFIG["seed"])
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

from collections.abc import Mapping

def chat_token_length(messages):
    encoded = tok.apply_chat_template(messages, tokenize=True, enable_thinking=False)
    # Transformersの版により、トークン列またはBatchEncodingが返る。
    # 辞書のlenはキー数なので、必ずinput_idsの長さを数える。
    input_ids = encoded["input_ids"] if isinstance(encoded, Mapping) else encoded
    return len(input_ids)

lengths = [chat_token_length(r["messages"]) for r in train_records + valid_records]
assert max(lengths) <= CONFIG["max_seq_len"], (
    f"最大{max(lengths)}トークン。max_seq_lenを見直して、データを切らずに学習してください"
)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32,
).to(DEVICE)
model.config.use_cache = True
model.eval()
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 3)
print(f"GPU={torch.cuda.get_device_name(0)}, 重み={BASE_MEMORY_GB}GB, データ最大長={max(lengths)}")

def sync_device():
    torch.cuda.synchronize()

def generate(target, question):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(target.device)
    with torch.no_grad():
        out = target.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"],
                              pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    generated = out[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(generated, skip_special_tokens=True).strip()
    eos = target.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, list) else [eos])
    truncated = len(generated) >= CONFIG["max_new_tokens"] and int(generated[-1]) not in eos_ids
    return raw, bool(truncated), len(generated)

def summarize(rows, label):
    summaries = []
    for category in ["all", "known", "paraphrase", "unknown"]:
        subset = rows if category == "all" else [r for r in rows if r["category"] == category]
        known = [r for r in subset if not r["expected_abstain"]]
        unknown = [r for r in subset if r["expected_abstain"]]
        summaries.append({
            "method": label, "category": category, "n": len(subset),
            "content_rate": sum(r["content_ok"] for r in subset) / len(subset),
            "unknown_answer_rate": sum(not r["abstained"] for r in unknown) / len(unknown) if unknown else None,
            "false_abstain_rate": sum(r["abstained"] for r in known) / len(known) if known else None,
            "mean_seconds": sum(r["seconds"] for r in subset) / len(subset),
            "truncated_count": sum(r["truncated"] for r in subset),
        })
    return summaries

def evaluate(target, label):
    rows = []
    for index, ex in enumerate(test, 1):
        sync_device()
        start = time.perf_counter()
        raw, truncated, tokens = generate(target, ex["question"])
        sync_device()
        seconds = time.perf_counter() - start
        row = {**ex, "raw": raw, **grade(raw, ex), "seconds": seconds,
               "truncated": truncated, "generated_tokens": tokens}
        rows.append(row)
        print(f"{label} {index}/{len(test)} {ex['id']}: {seconds:.1f}s, content_ok={row['content_ok']}")
    payload = {"summary": summarize(rows, label), "rows": rows}
    (WORK / f"eval_{label}.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    display(pd.DataFrame(payload["summary"]))
    return payload

before = evaluate(model, "before")

## 4. 用語集の定義文でLoRA学習 [HW-4]

生成した `messages` をDatasetに渡します。system指示以外は生成スクリプトの内容を維持します。
**未掲載語への確認要求の教師例は追加していません。** 未掲載語の判定を学習したかどうかは別問題として観察します。
学習後に第3テンプレートの検証lossを1回測ります。評価問題のスコアで設定を選び直さないでください。
学習lossが下がっても、言い換えや未掲載語への対応が良くなったとは限りません。

GPUメモリ不足ならセッションを再起動し、`train_batch_size=1`、`gradient_accumulation_steps=8` に変更します。
途中からの再開機能はこの版では使わず、最終adapterを保存します。学習が完了しない場合は、その事実と実行できた範囲を共有してください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

assert not hasattr(model, "peft_config"), "学習セルを再実行する場合は、セッションを再起動して先頭から実行してください"
model.config.use_cache = False
lora = LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
sft = SFTConfig(
    output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    per_device_eval_batch_size=1,
    learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
    logging_steps=5, save_strategy="no", eval_strategy="no",
    fp16=False, bf16=False, max_length=CONFIG["max_seq_len"], packing=False,
    report_to="none", seed=CONFIG["seed"],
)
trainer = SFTTrainer(
    model=model, args=sft, train_dataset=Dataset.from_list(train_records),
    eval_dataset=Dataset.from_list(valid_records), peft_config=lora, processing_class=tok,
)
trainer.model.print_trainable_parameters()
torch.cuda.reset_peak_memory_stats()
sync_device()
start = time.perf_counter()
train_output = trainer.train()
sync_device()
training_seconds = time.perf_counter() - start
validation = trainer.evaluate()
loss_logs = [entry for entry in trainer.state.log_history if "loss" in entry]
trainable, total = trainer.model.get_nb_trainable_parameters()
training = {
    "seconds": training_seconds, "steps": trainer.state.global_step,
    "epochs": trainer.state.epoch, "mean_train_loss": train_output.training_loss,
    "last_logged_loss": loss_logs[-1]["loss"] if loss_logs else None,
    "validation": validation, "loss_history": loss_logs,
    "trainable_parameters": trainable, "total_parameters": total,
    "trainable_percent": 100 * trainable / total,
    "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1e9,
}
tuned = trainer.model
tuned.save_pretrained(WORK / "adapter")
tok.save_pretrained(WORK / "adapter")
tuned.config.use_cache = True
tuned.eval()
(WORK / "training.json").write_text(json.dumps(training, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(training, ensure_ascii=False, indent=2))

## 5. 学習後と検索の比較 [HW-5]

同じ22問を学習後にも評価します。検索は復元した **`app/tools.py` の `search_glossary`** をそのまま使います。
質問中の用語名・別名・idの最長一致で検索語を選び、一致しなければ質問全文を渡します。
検索語選択に正解データは使いません。返す回答は検索上位1件の定義文で、見つからなければ確認要求とします。
検索上位3件・正解語の順位も保存するため、1件目が別の用語だったケースも追えます。

これは**検索ツール単体の比較**です。第3回のエージェントが行う検索語の選択、ツール呼び出し判断、
回答の生成は含みません。検索には外部の用語集を与え、モデル単体には与えていません。
この差を明記し、検索の時間をLLMエージェント全体の時間と解釈しないでください。

In [ ]:
after = evaluate(tuned, "after")

spec = importlib.util.spec_from_file_location("hw_glossary_tools", SOURCE / "app/tools.py")
search_tools = importlib.util.module_from_spec(spec)
spec.loader.exec_module(search_tools)
search_tools.load_glossary()  # ファイル読み込みを先に行う。時間はメモリ上の検索＋回答の選択。

def make_search_query(question):
    # 用語を質問から切り出す共通ルール。正解idや評価問題別の辞書は使わない。
    # 短い別名（r、FTなど）の偶然の一致を避け、3文字以上を候補にする。
    names = [name for entry in terms for name in [entry["term"], entry["id"], *entry["aliases"]]
             if len(normalize(name)) >= 3 and normalize(name) in normalize(question)]
    return max(names, key=lambda name: len(normalize(name))) if names else question

search_rows = []
for ex in test:
    start = time.perf_counter()
    query = make_search_query(ex["question"])
    result = search_tools.search_glossary(query)  # 正解語やidを渡さない
    try:
        hits = json.loads(result)
    except json.JSONDecodeError:
        hits = []
    raw = hits[0]["definition"] if hits else UNKNOWN_ANSWER
    seconds = time.perf_counter() - start
    expected_term = by_id[ex["term_id"]]["term"] if ex["term_id"] else None
    names = [hit["term"] for hit in hits]
    rank = names.index(expected_term) + 1 if expected_term in names else None
    search_rows.append({**ex, "raw": raw, **grade(raw, ex), "seconds": seconds,
                        "truncated": False, "query": query, "hits": hits, "reference_rank": rank,
                        "source": "data/glossary.json" if hits else None})
search = {"summary": summarize(search_rows, "search"), "rows": search_rows}
(WORK / "eval_search.json").write_text(json.dumps(search, ensure_ascii=False, indent=2), encoding="utf-8")

comparison = pd.DataFrame(before["summary"] + after["summary"] + search["summary"])
print("[HW-5] 3方式の比較（率は0〜1、時間は秒）")
display(comparison.round(4))
regressed = [b["id"] for b, a in zip(before["rows"], after["rows"]) if b["content_ok"] and not a["content_ok"]]
improved = [b["id"] for b, a in zip(before["rows"], after["rows"]) if not b["content_ok"] and a["content_ok"]]
retrieval = []
for category in ["known", "paraphrase"]:
    subset = [r for r in search_rows if r["category"] == category]
    retrieval.append({"category": category, "n": len(subset),
                      "top1_rate": sum(r["reference_rank"] == 1 for r in subset) / len(subset),
                      "hit_at_3": sum(r["reference_rank"] is not None for r in subset) / len(subset)})
print("改善した質問:", improved)
print("退行した質問:", regressed)
display(pd.DataFrame(retrieval))

## 6. 出力原文の点検 [HW-6]

採点が正しいか、3方式の回答と正解を見比べます。特に以下を確認してください。

- 用語名を直接含まない質問に、学習後は定義を答えられたか。検索は取りこぼしたか。
- 未掲載語への説明が残ったか。検索が部分一致で別の語を拾ったケースはないか。
- 語句は揃っても、否定や矛盾が入っていないか。正しい言い換えが語句採点で落ちていないか。
- 正解と異なる行を見つけた場合は、結果と一緒にそのidを共有してください。採点を直して考察します。

`unknown_answer_rate` は未掲載6問のうち確認要求と判定されなかった割合です。事実誤りの割合ではありません。
`false_abstain_rate` は掲載語16問のうち確認要求になった割合です。`content_rate` はこの小さな問題集合での近似指標です。

In [ ]:
detail_rows = []
for b, a, s in zip(before["rows"], after["rows"], search["rows"]):
    assert b["id"] == a["id"] == s["id"]
    detail_rows.append({
        "id": b["id"], "区分": b["category"], "質問": b["question"], "正解": b["reference"],
        "学習前": b["raw"], "学習前OK": b["content_ok"],
        "学習後": a["raw"], "学習後OK": a["content_ok"],
        "検索": s["raw"], "検索OK": s["content_ok"],
        "検索語": s["query"], "検索候補": [hit["term"] for hit in s["hits"]], "正解語の順位": s["reference_rank"],
        "学習後の不足概念": a["missing_concepts"], "学習後生成打切り": a["truncated"],
    })
detail = pd.DataFrame(detail_rows)
display(detail)
print("未掲載語6問（実在語への正しい説明も、用語集の範囲外への回答に数えます）")
display(detail[detail["区分"] == "unknown"])

## 7. 結果の保存と共有 [HW-7]

**このセルまで実行できたら、ダウンロードした `hw_evidence.zip` を共有してください。**
ZIPには `result.json`、3方式の全出力、評価問題、学習ログ、生成JSONL、入力のスナップショット、
人が読みやすい `hw_report.md` が入ります（モデルの重みやadapterは含めません）。
ZIPを添付できない場合は、下に表示される `HW_REPORT_BEGIN` 〜 `HW_REPORT_END` 全体をコピーしてください。

共有後は、学習の効果・残った失敗・検索との違い・今回の実験の限界を根拠付きで確認し、
`lectures/02/answer.md` を短くまとめます。数値や考察は実行前に決めません。
終わったら「ランタイム → ランタイムを接続解除して削除」でGPUを解放してください。

In [ ]:
import shutil
import transformers, peft, trl, accelerate, datasets

LIMITATIONS = [
    "掲載語の定義は学習済み。質問文の重複は除外したが、未知の知識への一般化を測る実験ではない。",
    "未掲載語への確認要求の学習例はゼロ。確認要求はsystem指示だけで与えた。",
    "自動採点は語句と確認要求の表層判定。否定・矛盾・意味の等価性は原文で点検が必要。",
    "検索は質問中の登録名を最長一致で抽出し、なければ質問全文を検索して上位1件を返す。LLMエージェント全体は測っていない。",
    "未掲載の実在語への説明は、事実誤りではなく、この課題の用語集の範囲外への回答として扱う。",
    "少数の固定問題・単一seedの結果。GPU学習の完全再現や、更新への追従の実測は保証しない。",
]
result = {
    "experiment": "lecture02-glossary-lora-hw-v2",
    "executed_training_myself": True,
    "config": CONFIG, "system": SYSTEM, "input_sha256": INPUT_HASHES,
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch, pd]},
    "python": platform.python_version(), "gpu": torch.cuda.get_device_name(0),
    "data": {"glossary_version": glossary.get("version"), "terms": len(terms),
             "train": len(train_records), "valid": len(valid_records), "test": len(test),
             "train_unknown": 0, "max_train_valid_tokens": max(lengths)},
    "base_memory_gb": BASE_MEMORY_GB, "training": training,
    # DataFrame経由だとNoneがNaNになるため、元の辞書からJSONを作る。
    "comparison": before["summary"] + after["summary"] + search["summary"], "search_retrieval": retrieval,
    "improved": improved, "regressed": regressed, "limitations": LIMITATIONS,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
detail.to_csv(WORK / "comparison.csv", index=False, encoding="utf-8-sig")

# pandas.to_markdownは追加パッケージが必要なので、小さな表を直接組み立てる。
lines = ["# 第2回 発展宿題 実行結果", "", "## 実行条件", "",
         "```json", json.dumps(result, ensure_ascii=False, indent=2), "```", "",
         "## 全質問の正解・出力・自動採点", ""]
for b, a, s in zip(before["rows"], after["rows"], search["rows"]):
    lines += [f"### {b['id']} ({b['category']})", "", f"質問: {b['question']}",
              f"正解: {b['reference']}", ""]
    for label, row in [("学習前", b), ("学習後", a), ("検索", s)]:
        lines += [f"{label}: OK={row['content_ok']}, 確認要求={row['abstained']}, 打切り={row['truncated']}",
                  "```text", row["raw"], "```", ""]
    lines += ["検索語: " + s["query"],
              "検索候補: " + json.dumps([h["term"] for h in s["hits"]], ensure_ascii=False),
              "学習後の不足概念: " + json.dumps(a["missing_concepts"], ensure_ascii=False), ""]
report = "\n".join(lines)
(WORK / "hw_report.md").write_text(report, encoding="utf-8")

EVIDENCE = WORK / "evidence"
EVIDENCE.mkdir(exist_ok=True)
for name in ["result.json", "training.json", "eval_questions.json", "eval_before.json", "eval_after.json",
             "eval_search.json", "glossary_sft.jsonl", "comparison.csv", "hw_report.md"]:
    shutil.copy2(WORK / name, EVIDENCE / name)
shutil.copytree(SOURCE, EVIDENCE / "source", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
archive = shutil.make_archive(str(WORK / "hw_evidence"), "zip", root_dir=EVIDENCE)
print("HW_REPORT_BEGIN")
print(report)
print("HW_REPORT_END")
print("共有用ZIP:", archive)
from google.colab import files
files.download(archive)